# Занятие 10. Диффузионные модели (DDPM / LDM)

## Цели занятия:
- Изучить принцип работы диффузионных моделей (Forward/Reverse process)
- Реализовать упрощённый DDPM (Denoising Diffusion Probabilistic Model)
- Обучить модель предсказания шума на датасете MNIST
- Сравнить качество генерации с VAE и GAN

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import random
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Шаг 1. Подготовка данных и гиперпараметров

In [ ]:
# Diffusion hyperparameters
num_steps = 100  # T - number of diffusion steps
beta_start = 0.0001
beta_end = 0.02

# Linear beta schedule
betas = torch.linspace(beta_start, beta_end, num_steps).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

# Data
transform = transforms.Compose([transforms.ToTensor()])
dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0)

print(f"Number of diffusion steps: {num_steps}")
print(f"Beta range: [{beta_start}, {beta_end}]")
print(f"Dataset size: {len(dataset)}")

---
## Шаг 2. Функции добавления шума (Forward Process)

In [ ]:
def extract(a, t, x_shape):
    """Extract coefficients at t and reshape for broadcasting.
    
    Args:
        a: Tensor of shape (T,) - coefficients
        t: Tensor of shape (B,) - timestep indices
        x_shape: Shape of input tensor for broadcasting
    """
    batch_size = t.shape[0]
    out = a.gather(-1, t.cpu()).to(t.device)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))


def q_sample(x_start, t, noise=None):
    """Forward diffusion process: add noise to x_start at timestep t.
    
    x_t = sqrt(alpha_cumprod_t) * x_start + sqrt(1 - alpha_cumprod_t) * noise
    
    This is the closed-form solution for adding noise at any timestep.
    """
    if noise is None:
        noise = torch.randn_like(x_start)
    
    sqrt_alphas_cumprod_t = extract(alphas_cumprod.sqrt(), t, x_start.shape)
    sqrt_one_minus_alphas_cumprod_t = extract(
        (1.0 - alphas_cumprod).sqrt(), t, x_start.shape
    )
    
    return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise

In [ ]:
# Visualize forward process
def visualize_forward_process():
    """Show how noise is added over timesteps."""
    img, _ = dataset[0]
    x_start = img.unsqueeze(0).to(device)
    
    timesteps = [0, 20, 40, 60, 80, 99]
    
    fig, axes = plt.subplots(1, len(timesteps), figsize=(15, 3))
    
    for i, t in enumerate(timesteps):
        t_tensor = torch.tensor([t]).to(device)
        x_t = q_sample(x_start, t_tensor)
        
        axes[i].imshow(x_t.squeeze().cpu(), cmap='gray')
        axes[i].set_title(f't={t}')
        axes[i].axis('off')
    
    plt.suptitle('Forward Diffusion Process (Adding Noise)', fontsize=14)
    plt.tight_layout()
    plt.savefig('diffusion_forward.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_forward_process()

---
## Шаг 3. Модель UNet (Упрощённая)

In [ ]:
class SinusoidalPositionEmbedding(nn.Module):
    """Sinusoidal positional embedding for timesteps."""
    
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    
    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        embeddings = np.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = t.unsqueeze(-1) * embeddings.unsqueeze(0)
        embeddings = torch.cat([torch.sin(embeddings), torch.cos(embeddings)], dim=-1)
        return embeddings


class SimpleUNet(nn.Module):
    """Simplified UNet for DDPM.
    
    In practice, use a full UNet with skip connections.
    This is a simplified version for educational purposes.
    """
    
    def __init__(self, num_steps=100, embed_dim=128):
        super().__init__()
        
        # Time embedding
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(embed_dim),
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU()
        )
        
        # Encoder
        self.enc1 = nn.Conv2d(1, 32, 3, padding=1)
        self.enc2 = nn.Conv2d(32, 64, 3, padding=1)
        self.enc3 = nn.Conv2d(64, 128, 3, padding=1)
        
        # Time projection
        self.time_proj = nn.Linear(embed_dim, 128)
        
        # Decoder
        self.dec3 = nn.Conv2d(128, 64, 3, padding=1)
        self.dec2 = nn.Conv2d(128, 32, 3, padding=1)  # 64 + 64 skip
        self.dec1 = nn.Conv2d(64, 1, 3, padding=1)    # 32 + 32 skip
        
        self.pool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
    
    def forward(self, x, t):
        # Time embedding
        t_emb = self.time_embed(t)
        t_proj = self.time_proj(t_emb).unsqueeze(-1).unsqueeze(-1)
        
        # Encoder
        e1 = F.relu(self.enc1(x))
        e2 = F.relu(self.enc2(self.pool(e1)))
        e3 = F.relu(self.enc3(self.pool(e2)))
        
        # Add time embedding
        e3 = e3 + t_proj
        
        # Decoder with skip connections
        d3 = F.relu(self.dec3(self.upsample(e3)))
        d2 = F.relu(self.dec2(torch.cat([d3, e2], dim=1)))
        d1 = self.dec1(torch.cat([self.upsample(d2), e1], dim=1))
        
        return d1

# Create model
model = SimpleUNet(num_steps=100).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## Шаг 4. Цикл обучения

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
history = []

print("="*50)
print("Training Diffusion Model (DDPM)")
print("="*50)

for epoch in range(10):
    model.train()
    total_loss = 0
    
    for x_start, _ in loader:
        x_start = x_start.to(device)
        batch_size = x_start.shape[0]
        
        # Sample random timesteps
        t = torch.randint(0, num_steps, (batch_size,), device=device).long()
        
        # Generate noise
        noise = torch.randn_like(x_start)
        
        # Add noise (forward process)
        x_noisy = q_sample(x_start, t, noise)
        
        # Predict noise
        noise_pred = model(x_noisy, t)
        
        # Loss: MSE between predicted and actual noise
        loss = F.mse_loss(noise_pred, noise)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(loader)
    history.append(avg_loss)
    print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}")

---
## Шаг 5. Процесс сэмплирования (Генерация)

In [ ]:
@torch.no_grad()
def p_sample(model, x, t):
    """Single reverse diffusion step.
    
    Predict noise and remove it from x_t to get x_{t-1}.
    """
    # Predict noise
    noise_pred = model(x, t)
    
    # Get coefficients
    alpha_t = alphas[t[0]]
    alpha_cumprod_t = alphas_cumprod[t[0]]
    
    # Compute mean
    mean = (1 / alpha_t.sqrt()) * (x - (1 - alpha_t) / (1 - alpha_cumprod_t).sqrt() * noise_pred)
    
    # Add noise if not last step
    if t[0] > 0:
        noise = torch.randn_like(x)
        sigma = betas[t[0]].sqrt()
        mean = mean + sigma * noise
    
    return mean


@torch.no_grad()
def sample(model, num_samples=16):
    """Generate samples by reverse diffusion.
    
    Start from pure noise and iteratively denoise.
    """
    model.eval()
    
    # Start from noise
    x = torch.randn(num_samples, 1, 28, 28).to(device)
    
    # Iteratively denoise
    for t in reversed(range(num_steps)):
        t_batch = torch.full((num_samples,), t, device=device, dtype=torch.long)
        x = p_sample(model, x, t_batch)
    
    # Clamp to valid range
    x = x.clamp(0, 1)
    return x

In [ ]:
# Generate samples
print("Generating samples...")
samples = sample(model, num_samples=16)

# Visualize
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].squeeze().cpu(), cmap='gray')
    ax.axis('off')

plt.suptitle('Diffusion Generated Images', fontsize=14)
plt.tight_layout()
plt.savefig('diffusion_generated.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Шаг 6. Сравнение генеративных моделей

In [ ]:
import pandas as pd

comparison = {
    'Model': ['AE', 'VAE', 'GAN', 'Diffusion'],
    'Speed': ['Fast', 'Fast', 'Fast', 'Slow (iterative)'],
    'Quality': ['Low', 'Medium', 'High', 'Very High'],
    'Stability': ['Stable', 'Stable', 'Unstable', 'Stable'],
    'Diversity': ['Low', 'High', 'Medium', 'Very High'],
    'Latent Space': ['Discrete', 'Continuous', 'Not interpretable', 'Not interpretable']
}

df = pd.DataFrame(comparison)
print("\n" + "="*60)
print("Generative Models Comparison")
print("="*60)
print(df.to_string(index=False))

In [ ]:
# Training loss curve
plt.figure(figsize=(10, 5))
plt.plot(history, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Diffusion Model Training Loss')
plt.grid(True, alpha=0.3)
plt.savefig('diffusion_loss.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Домашнее задание

1. Увеличить число шагов диффузии до 500 и оценить качество
2. Реализовать косинусное расписание шума (cosine schedule)
3. Подготовить презентацию по Мини-проекту

---
## Контрольные вопросы

1. Зачем нужен обратный процесс в диффузии?
2. Почему обучение диффузии стабильнее чем GAN?
3. Как влияет количество шагов диффузии на качество генерации?
4. В чем преимущество DDPM перед VAE?